In [22]:
import numpy as np
import random
import mediapipe as mp
import json
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
from collections import defaultdict
import shutil
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping
import cv2
import tensorflow as tf

In [2]:
 #── Config ────────────────────────────────────────────────────────────────────
input_split_folder  = r'C:\Users\justi\MachineLearning\Project\dataset_split'
output_split_folder = r'C:\Users\justi\MachineLearning\Project\dataset_split_mpLM_new15'
TARGET_FRAMES = 15

# ── MediaPipe Setup ───────────────────────────────────────────────────────────
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.3,
    min_tracking_confidence=0.3
)

# ── Landmark Extraction ───────────────────────────────────────────────────────
def extract_landmarks(video_path, target_frames=15):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return None

    # Sample 2x frames to account for dropped detections
    indices = np.linspace(0, total_frames - 1, target_frames * 2, dtype=int)
    valid_landmarks = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)

        # Only keep frames where at least one hand was detected
        if results.multi_hand_landmarks:
            frame_landmarks = np.zeros(21 * 3 * 2)  # 2 hands x 21 landmarks x xyz
            for i, hand_lm in enumerate(results.multi_hand_landmarks):
                if i >= 2:
                    break
                for j, lm in enumerate(hand_lm.landmark):
                    frame_landmarks[i * 63 + j * 3]     = lm.x
                    frame_landmarks[i * 63 + j * 3 + 1] = lm.y
                    frame_landmarks[i * 63 + j * 3 + 2] = lm.z
            valid_landmarks.append(frame_landmarks)

        if len(valid_landmarks) >= target_frames:
            break

    cap.release()

    if len(valid_landmarks) == 0:
        return None  # No hands detected at all — skip this video

    # Resample to exactly target_frames
    idx = np.linspace(0, len(valid_landmarks) - 1, target_frames, dtype=int)
    return np.array(valid_landmarks)[idx]  # shape: (target_frames, 126)


# ── Process All Splits ────────────────────────────────────────────────────────
skipped_log  = defaultdict(list)
saved_counts = defaultdict(int)

for split in ['train', 'val', 'test']:
    split_path = os.path.join(input_split_folder, split)
    if not os.path.exists(split_path):
        continue

    print(f'\n=== Processing {split} ===')

    for class_name in sorted(os.listdir(split_path)):
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue

        out_folder = os.path.join(output_split_folder, split, class_name)
        os.makedirs(out_folder, exist_ok=True)

        videos = [f for f in os.listdir(class_path)
                  if f.endswith(('.mp4', '.mkv', '.webm', '.avi'))]

        for video_file in videos:
            video_path = os.path.join(class_path, video_file)
            video_id   = os.path.splitext(video_file)[0]
            out_path   = os.path.join(out_folder, f'{video_id}.npy')

            if os.path.exists(out_path):
                print(f'  Skipping {video_id} — already processed')
                saved_counts[class_name] += 1
                continue

            try:
                landmarks = extract_landmarks(video_path, TARGET_FRAMES)

                if landmarks is None:
                    print(f'  SKIPPED {video_id} — no hand detections')
                    skipped_log[class_name].append(video_id)
                    continue

                assert landmarks.shape == (TARGET_FRAMES, 126), f'Wrong shape: {landmarks.shape}'
                assert not np.all(landmarks == 0, axis=1).all(), f'All frames are zero: {video_id}'
                
                np.save(out_path, landmarks)
                saved_counts[class_name] += 1
                print(f'  Saved {video_id} — shape {landmarks.shape}')
            
            except AssertionError as e:
                print(f'  FAILED assertion {video_id}: {e}')
                skipped_log[class_name].append(video_id)

            
            except Exception as e:
                print(f'  FAILED {video_id}: {e}')
                skipped_log[class_name].append(video_id)

            
            

hands.close()

# ── Summary ───────────────────────────────────────────────────────────────────
print('\n=== Skipped Videos Per Class ===')
for class_name, skipped in sorted(skipped_log.items()):
    print(f'  {class_name}: {len(skipped)} skipped')

print('\n=== Saved Videos Per Class ===')
for class_name, count in sorted(saved_counts.items()):
    print(f'  {class_name}: {count} saved')

total_skipped = sum(len(v) for v in skipped_log.values())
total_saved   = sum(saved_counts.values())
print(f'\nTotal saved:   {total_saved}')
print(f'Total skipped: {total_skipped}')


=== Processing train ===
  Saved 00623 — shape (15, 126)
  Saved 00624 — shape (15, 126)
  Saved 00625 — shape (15, 126)
  Saved 00628 — shape (15, 126)
  Saved 00629 — shape (15, 126)
  SKIPPED 00639 — no hand detections
  Saved 01384 — shape (15, 126)
  Saved 01386 — shape (15, 126)
  Saved 01387 — shape (15, 126)
  Saved 01388 — shape (15, 126)
  SKIPPED 01398 — no hand detections
  Saved 01418 — shape (15, 126)
  Saved 01420 — shape (15, 126)
  Saved 01422 — shape (15, 126)
  SKIPPED 01428 — no hand detections
  Saved 01460 — shape (15, 126)
  Saved 01462 — shape (15, 126)
  Saved 01464 — shape (15, 126)
  SKIPPED 01471 — no hand detections
  Saved 02999 — shape (15, 126)
  Saved 03000 — shape (15, 126)
  Saved 03002 — shape (15, 126)
  Saved 03003 — shape (15, 126)
  SKIPPED 03008 — no hand detections
  Saved 03055 — shape (15, 126)
  Saved 03056 — shape (15, 126)
  Saved 03058 — shape (15, 126)
  Saved 03059 — shape (15, 126)
  Saved 03060 — shape (15, 126)
  Saved 03119 — shape

In [5]:
MIN_TRAIN = 4
MIN_VAL   = 1
MIN_TEST  = 1

print('=== Checking class minimums ===')
classes_to_remove = []

# Get all classes from train
train_path = os.path.join(output_split_folder, 'train')
for class_name in os.listdir(train_path):
    class_path = os.path.join(train_path, class_name)
    if not os.path.isdir(class_path):
        continue

    train_count = len([f for f in os.listdir(class_path) if f.endswith('.npy')])

    val_path  = os.path.join(output_split_folder, 'val', class_name)
    test_path = os.path.join(output_split_folder, 'test', class_name)

    val_count  = len([f for f in os.listdir(val_path)  if f.endswith('.npy')]) if os.path.exists(val_path)  else 0
    test_count = len([f for f in os.listdir(test_path) if f.endswith('.npy')]) if os.path.exists(test_path) else 0

    if train_count < MIN_TRAIN or val_count < MIN_VAL or test_count < MIN_TEST:
        classes_to_remove.append(class_name)
        print(f'  REMOVE {class_name}: train={train_count} val={val_count} test={test_count}')
    else:
        print(f'  KEEP   {class_name}: train={train_count} val={val_count} test={test_count}')


# Remove failing classes from all splits
print(f'\n=== Removing {len(classes_to_remove)} classes ===')
for class_name in classes_to_remove:
    for split in ['train', 'val', 'test']:
        class_path = os.path.join(output_split_folder, split, class_name)
        if os.path.exists(class_path):
            shutil.rmtree(class_path)
            print(f'  Removed {class_name} from {split}')

# Final summary
remaining = [c for c in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, c))]
print(f'\nTotal classes remaining: {len(remaining)}')



=== Checking class minimums ===
  KEEP   accident: train=5 val=1 test=1
  KEEP   africa: train=4 val=1 test=1
  REMOVE after: train=3 val=1 test=1
  REMOVE again: train=3 val=1 test=1
  KEEP   apple: train=4 val=1 test=1
  REMOVE appointment: train=5 val=0 test=1
  REMOVE approve: train=4 val=1 test=0
  REMOVE arrive: train=4 val=0 test=1
  KEEP   australia: train=4 val=1 test=1
  REMOVE awkward: train=3 val=1 test=1
  REMOVE backpack: train=3 val=1 test=1
  KEEP   bad: train=4 val=1 test=1
  REMOVE balance: train=3 val=1 test=1
  REMOVE ball: train=3 val=1 test=1
  REMOVE banana: train=3 val=1 test=1
  REMOVE bar: train=4 val=0 test=1
  KEEP   basketball: train=5 val=1 test=1
  KEEP   beard: train=4 val=1 test=1
  REMOVE because: train=4 val=0 test=1
  KEEP   bed: train=4 val=1 test=1
  REMOVE beer: train=3 val=1 test=1
  KEEP   before: train=5 val=2 test=1
  KEEP   bird: train=5 val=1 test=1
  REMOVE bitter: train=3 val=1 test=1
  REMOVE black: train=3 val=1 test=1
  REMOVE blanket: 

In [10]:
# ── 1. Load Data ──────────────────────────────────────────────────────────────
def load_split(split_folder, split, valid_classes):
    X, y = [], []
    split_path = os.path.join(split_folder, split)

    for class_name in valid_classes:
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue
        for npy_file in os.listdir(class_path):
            if not npy_file.endswith('.npy'):
                continue
            data = np.load(os.path.join(class_path, npy_file))
            X.append(data)
            y.append(class_name)

    return np.array(X), np.array(y)

X_train, y_train = load_split(output_split_folder, 'train', valid_classes)
X_val,   y_val   = load_split(output_split_folder, 'val',   valid_classes)
X_test,  y_test  = load_split(output_split_folder, 'test',  valid_classes)

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')

# ── 2. Encode Labels ──────────────────────────────────────────────────────────
le = LabelEncoder()
le.fit(y_train)

y_train_enc = to_categorical(le.transform(y_train))
y_val_enc   = to_categorical(le.transform(y_val))
y_test_enc  = to_categorical(le.transform(y_test))

num_classes = len(le.classes_)
print(f'Classes: {num_classes}')

# ── 3. Build Model ────────────────────────────────────────────────────────────
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(15, 126)),
    Dropout(0.5),
    LSTM(64),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

# ── 4. Compile ────────────────────────────────────────────────────────────────
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── 5. Train ──────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True)

history = model.fit(
    X_train, y_train_enc,
    validation_data=(X_val, y_val_enc),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop]
)

# ── 6. Evaluate ───────────────────────────────────────────────────────────────
loss, accuracy = model.evaluate(X_test, y_test_enc)
print(f'\nTest Accuracy: {accuracy * 100:.2f}%')

X_train: (355, 15, 126)
X_val:   (84, 15, 126)
X_test:  (84, 15, 126)
Classes: 79


C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                        │ (None, 15, 64)              │          48,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 15, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_7 (LSTM)                        │ (None, 64)                  │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 79)                  │           5,135 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 91,215 (356.31 KB)

 Trainable params: 91,215 (356.31 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.0141 - loss: 4.3828 - val_accuracy: 0.0476 - val_loss: 4.3443
Epoch 2/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0197 - loss: 4.3468 - val_accuracy: 0.0357 - val_loss: 4.3076
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0254 - loss: 4.2749 - val_accuracy: 0.0238 - val_loss: 4.2401
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0366 - loss: 4.1673 - val_accuracy: 0.0238 - val_loss: 4.1735
Epoch 5/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0507 - loss: 4.1086 - val_accuracy: 0.0238 - val_loss: 4.1217
Epoch 6/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0310 - loss: 4.0614 - val_accuracy: 0.0119 - val_loss: 4.0888
Epoch 7/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0394 - loss: 3.9664 - val_accuracy: 0.0238 - val_loss: 4.0057
Epoch 8/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0535 - loss: 3.9388 - val_accuracy: 0.

In [13]:
from collections import Counter
class_counts = Counter(y_train)
top_n = 10  # try 10, 15, 20
top_classes = [c for c, _ in class_counts.most_common(top_n)]

# 2. Retrain using only those classes
X_train_sub = X_train[np.isin(y_train, top_classes)]
y_train_sub = y_train[np.isin(y_train, top_classes)]

# Filter val and test to only top_classes
X_val_sub  = X_val[np.isin(y_val, top_classes)]
y_val_sub  = y_val[np.isin(y_val, top_classes)]
X_test_sub = X_test[np.isin(y_test, top_classes)]
y_test_sub = y_test[np.isin(y_test, top_classes)]

# Re-encode labels for only these classes
le_sub = LabelEncoder()
le_sub.fit(y_train_sub)

y_train_enc_sub = to_categorical(le_sub.transform(y_train_sub))
y_val_enc_sub   = to_categorical(le_sub.transform(y_val_sub))
y_test_enc_sub  = to_categorical(le_sub.transform(y_test_sub))

num_classes_sub = len(le_sub.classes_)
print(f'Classes: {num_classes_sub}')
print(f'Train: {X_train_sub.shape}')
print(f'Val:   {X_val_sub.shape}')
print(f'Test:  {X_test_sub.shape}')
print(f'Random chance: {1/num_classes_sub*100:.2f}%')

# Rebuild model for new number of classes
model_sub = Sequential([
    LSTM(64, return_sequences=True, input_shape=(15, 126)),
    Dropout(0.5),
    LSTM(64),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(num_classes_sub, activation='softmax')
])

model_sub.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model_sub.fit(
    X_train_sub, y_train_enc_sub,
    validation_data=(X_val_sub, y_val_enc_sub),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop]
)

loss, accuracy = model_sub.evaluate(X_test_sub, y_test_enc_sub)
print(f'\nTest Accuracy: {accuracy * 100:.2f}%')
print(f'Random chance: {1/num_classes_sub*100:.2f}%')

Classes: 10
Train: (59, 15, 126)
Val:   (15, 15, 126)
Test:  (10, 15, 126)
Random chance: 10.00%
Epoch 1/100


C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 173ms/step - accuracy: 0.1186 - loss: 2.3032 - val_accuracy: 0.0667 - val_loss: 2.2656
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.1186 - loss: 2.2532 - val_accuracy: 0.1333 - val_loss: 2.2295
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.1864 - loss: 2.2075 - val_accuracy: 0.1333 - val_loss: 2.1961
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.1356 - loss: 2.1675 - val_accuracy: 0.1333 - val_loss: 2.1722
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.1695 - loss: 2.1331 - val_accuracy: 0.2000 - val_loss: 2.1590
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.2373 - loss: 2.0866 - val_accuracy: 0.2000 - val_loss: 2.1485
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2373 - loss: 2.0423 - val_accuracy: 0.2000 - val_loss: 2.1351
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.1186 - loss: 2.0598 - val_accuracy: 0.2000 - val_loss: 2.1101
Epo

In [12]:
from sklearn.metrics import classification_report

y_pred = model_sub.predict(X_test_sub)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test_enc_sub, axis=1)

print(classification_report(
    y_test_classes,
    y_pred_classes,
    target_names=le_sub.classes_
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step
              precision    recall  f1-score   support

    accident       0.00      0.00      0.00         1
  basketball       0.50      1.00      0.67         1
      before       1.00      1.00      1.00         1
        bird       0.50      1.00      0.67         1
      carrot       0.50      1.00      0.67         1
    computer       1.00      1.00      1.00         1
        cool       0.00      0.00      0.00         1
        thin       0.00      0.00      0.00         1
       trade       1.00      1.00      1.00         1
         who       1.00      1.00      1.00         1

    accuracy                           0.70        10
   macro avg       0.55      0.70      0.60        10
weighted avg       0.55      0.70      0.60        10



C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [14]:
print("=== Sample counts per class (train) ===")
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f'  {cls}: {count}')

print(f'\nMax samples in any class: {max(class_counts.values())}')
print(f'Min samples in any class: {min(class_counts.values())}')
print(f'Average: {sum(class_counts.values())/len(class_counts):.1f}')

=== Sample counts per class (train) ===
  computer: 7
  thin: 7
  trade: 7
  carrot: 6
  cool: 6
  who: 6
  accident: 5
  basketball: 5
  before: 5
  bird: 5
  bowling: 5
  brother: 5
  change: 5
  cold: 5
  corn: 5
  dark: 5
  family: 5
  hot: 5
  later: 5
  laugh: 5
  leave: 5
  new: 5
  pizza: 5
  school: 5
  tall: 5
  thanksgiving: 5
  what: 5
  white: 5
  write: 5
  yes: 5
  africa: 4
  apple: 4
  australia: 4
  bad: 4
  beard: 4
  bed: 4
  cloud: 4
  cookie: 4
  cow: 4
  day: 4
  dive: 4
  drink: 4
  example: 4
  fish: 4
  football: 4
  girl: 4
  go: 4
  good: 4
  graduate: 4
  hearing: 4
  hope: 4
  how: 4
  letter: 4
  lose: 4
  mother: 4
  necklace: 4
  no: 4
  outside: 4
  pepper: 4
  pink: 4
  pull: 4
  shirt: 4
  short: 4
  soft: 4
  speech: 4
  study: 4
  surgery: 4
  tiger: 4
  tired: 4
  toast: 4
  visit: 4
  vomit: 4
  wait: 4
  want: 4
  why: 4
  woman: 4
  work: 4
  yesterday: 4
  you: 4

Max samples in any class: 7
Min samples in any class: 4
Average: 4.5


In [16]:
from sklearn.model_selection import StratifiedKFold
# Combine all data across splits
X_all = np.concatenate([X_train_sub, X_val_sub, X_test_sub], axis=0)
y_all = np.concatenate([y_train_sub, y_val_sub, y_test_sub], axis=0)

# Re-encode all labels
le_all = LabelEncoder()
y_all_enc = le_all.fit_transform(y_all)

print(f'Total samples: {len(X_all)}')
print(f'Total classes: {len(le_all.classes_)}')
print(f'Random chance: {1/len(le_all.classes_)*100:.1f}%')

# 5-fold cross validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(X_all, y_all_enc)):
    print(f'\n=== Fold {fold+1}/5 ===')

    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all_enc[train_idx], y_all_enc[test_idx]

    y_tr_cat = to_categorical(y_tr, num_classes=len(le_all.classes_))
    y_te_cat = to_categorical(y_te, num_classes=len(le_all.classes_))

    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(15, 126)),
        Dropout(0.5),
        LSTM(64),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(len(le_all.classes_), activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    model.fit(
        X_tr, y_tr_cat,
        validation_split=0.2,
        epochs=100,
        batch_size=8,
        callbacks=[early_stop],
        verbose=0
    )

    _, acc = model.evaluate(X_te, y_te_cat, verbose=0)
    fold_accuracies.append(acc)
    print(f'  Fold {fold+1} accuracy: {acc*100:.2f}%')

print(f'\n=== Cross Validation Results ===')
print(f'Per fold: {[f"{a*100:.1f}%" for a in fold_accuracies]}')
print(f'Mean accuracy: {np.mean(fold_accuracies)*100:.2f}%')
print(f'Std:           {np.std(fold_accuracies)*100:.2f}%')
print(f'Random chance: {1/len(le_all.classes_)*100:.1f}%')

Total samples: 84
Total classes: 10
Random chance: 10.0%

=== Fold 1/5 ===


C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Fold 1 accuracy: 11.76%

=== Fold 2/5 ===
  Fold 2 accuracy: 35.29%

=== Fold 3/5 ===
  Fold 3 accuracy: 41.18%

=== Fold 4/5 ===
  Fold 4 accuracy: 35.29%

=== Fold 5/5 ===
  Fold 5 accuracy: 37.50%

=== Cross Validation Results ===
Per fold: ['11.8%', '35.3%', '41.2%', '35.3%', '37.5%']
Mean accuracy: 32.21%
Std:           10.44%
Random chance: 10.0%


In [18]:

# ── Config ────────────────────────────────────────────────────────────────────
input_split_folder  = r'C:\Users\justi\MachineLearning\Project\dataset_split'
output_split_folder = r'C:\Users\justi\MachineLearning\Project\dataset_split_mpLM_aug2'
TARGET_FRAMES = 10

# Only these classes have enough samples
VALID_CLASSES = {
    'computer', 'thin', 'trade', 'carrot', 'cool', 'who', 'accident',
    'basketball', 'before', 'bird', 'bowling', 'brother', 'change', 'cold',
    'corn', 'dark', 'family', 'hot', 'later', 'laugh', 'leave', 'new',
    'pizza', 'school', 'tall', 'thanksgiving', 'what', 'white', 'write', 'yes'
}

# Train gets all 3 speeds, val/test get normal only
TRAIN_SPEEDS = [(1.0, ''), (0.75, '_slow'), (1.25, '_fast')]
EVAL_SPEEDS  = [(1.0, '')]

# ── MediaPipe Setup ───────────────────────────────────────────────────────────
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.3,
    min_tracking_confidence=0.3
)

# ── Landmark Extraction ───────────────────────────────────────────────────────
def extract_landmarks(video_path, target_frames=10, speed=1.0):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return None

    # speed=0.75 -> sample fewer source frames (slow motion effect)
    # speed=1.0  -> sample all frames (normal)
    # speed=1.25 -> sample more source frames (fast forward effect)
    n_source_frames = int(total_frames * speed)
    n_source_frames = np.clip(n_source_frames, target_frames, total_frames)

    indices = np.linspace(0, n_source_frames - 1, target_frames * 2, dtype=int)
    indices = np.clip(indices, 0, total_frames - 1)

    valid_landmarks = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)

        if results.multi_hand_landmarks:
            frame_landmarks = np.zeros(21 * 3 * 2)
            for i, hand_lm in enumerate(results.multi_hand_landmarks):
                if i >= 2:
                    break
                for j, lm in enumerate(hand_lm.landmark):
                    frame_landmarks[i * 63 + j * 3]     = lm.x
                    frame_landmarks[i * 63 + j * 3 + 1] = lm.y
                    frame_landmarks[i * 63 + j * 3 + 2] = lm.z
            valid_landmarks.append(frame_landmarks)

        if len(valid_landmarks) >= target_frames:
            break

    cap.release()

    if len(valid_landmarks) == 0:
        return None

    idx = np.linspace(0, len(valid_landmarks) - 1, target_frames, dtype=int)
    return np.array(valid_landmarks)[idx]  # shape: (target_frames, 126)


# ── Process All Splits ────────────────────────────────────────────────────────
skipped_log  = defaultdict(list)
saved_counts = defaultdict(int)

for split in ['train', 'val', 'test']:
    split_path = os.path.join(input_split_folder, split)
    if not os.path.exists(split_path):
        continue

    print(f'\n=== Processing {split} ===')

    # Pick speeds based on split
    speeds = TRAIN_SPEEDS if split == 'train' else EVAL_SPEEDS

    for class_name in sorted(os.listdir(split_path)):

        # Skip classes not in our valid set
        if class_name not in VALID_CLASSES:
            continue

        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue

        out_folder = os.path.join(output_split_folder, split, class_name)
        os.makedirs(out_folder, exist_ok=True)

        videos = [f for f in os.listdir(class_path)
                  if f.endswith(('.mp4', '.mkv', '.webm', '.avi'))]

        for video_file in videos:
            video_path = os.path.join(class_path, video_file)
            video_id   = os.path.splitext(video_file)[0]

            for speed, suffix in speeds:
                out_path = os.path.join(out_folder, f'{video_id}{suffix}.npy')

                if os.path.exists(out_path):
                    print(f'  Skipping {video_id}{suffix} — already processed')
                    saved_counts[class_name] += 1
                    continue

                try:
                    landmarks = extract_landmarks(video_path, TARGET_FRAMES, speed)

                    if landmarks is None:
                        print(f'  SKIPPED {video_id}{suffix} — no hand detections')
                        skipped_log[class_name].append(f'{video_id}{suffix}')
                        continue

                    assert landmarks.shape == (TARGET_FRAMES, 126), \
                        f'Wrong shape: {landmarks.shape}'
                    assert not np.all(landmarks == 0, axis=1).all(), \
                        f'All frames are zero: {video_id}{suffix}'

                    np.save(out_path, landmarks)
                    saved_counts[class_name] += 1
                    print(f'  Saved {video_id}{suffix} — shape {landmarks.shape}')

                except AssertionError as e:
                    print(f'  FAILED assertion {video_id}{suffix}: {e}')
                    skipped_log[class_name].append(f'{video_id}{suffix}')
                except Exception as e:
                    print(f'  FAILED {video_id}{suffix}: {e}')
                    skipped_log[class_name].append(f'{video_id}{suffix}')

hands.close()

# ── Summary ───────────────────────────────────────────────────────────────────
print('\n=== Skipped Per Class ===')
for class_name, skipped in sorted(skipped_log.items()):
    print(f'  {class_name}: {len(skipped)} skipped')

print('\n=== Saved Per Class (all splits) ===')
for class_name, count in sorted(saved_counts.items()):
    print(f'  {class_name}: {count} saved')

total_skipped = sum(len(v) for v in skipped_log.values())
total_saved   = sum(saved_counts.values())
print(f'\nTotal saved:   {total_saved}')
print(f'Total skipped: {total_skipped}')


=== Processing train ===
  Saved 00623 — shape (10, 126)
  Saved 00623_slow — shape (10, 126)
  Saved 00623_fast — shape (10, 126)
  Saved 00624 — shape (10, 126)
  Saved 00624_slow — shape (10, 126)
  Saved 00624_fast — shape (10, 126)
  Saved 00625 — shape (10, 126)
  Saved 00625_slow — shape (10, 126)
  Saved 00625_fast — shape (10, 126)
  Saved 00628 — shape (10, 126)
  Saved 00628_slow — shape (10, 126)
  Saved 00628_fast — shape (10, 126)
  Saved 00629 — shape (10, 126)
  Saved 00629_slow — shape (10, 126)
  Saved 00629_fast — shape (10, 126)
  SKIPPED 00639 — no hand detections
  SKIPPED 00639_slow — no hand detections
  SKIPPED 00639_fast — no hand detections
  Saved 05231 — shape (10, 126)
  Saved 05231_slow — shape (10, 126)
  Saved 05231_fast — shape (10, 126)
  Saved 05232 — shape (10, 126)
  Saved 05232_slow — shape (10, 126)
  Saved 05232_fast — shape (10, 126)
  Saved 05233 — shape (10, 126)
  Saved 05233_slow — shape (10, 126)
  Saved 05233_fast — shape (10, 126)
  Sav

In [27]:

# ── Random Seed ───────────────────────────────────────────────────────────────
D = 80
random.seed(D)
np.random.seed(D)
tf.random.set_seed(D)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
# ── Config ────────────────────────────────────────────────────────────────────
output_split_folder = r'C:\Users\justi\MachineLearning\Project\dataset_split_mpLM_aug2'
TARGET_FRAMES = 10

VALID_CLASSES = [
    'computer', 'thin', 'trade', 'carrot', 'cool', 'who', 'accident',
    'basketball', 'before', 'bird', 'bowling', 'brother', 'change', 'cold',
    'corn', 'dark', 'family', 'hot', 'later', 'laugh', 'leave', 'new',
    'pizza', 'school', 'tall', 'thanksgiving', 'what', 'white', 'write', 'yes'
]

# ── 1. Load Data ──────────────────────────────────────────────────────────────
def load_split(split_folder, split, valid_classes):
    X, y = [], []
    split_path = os.path.join(split_folder, split)
    for class_name in valid_classes:
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue
        for npy_file in os.listdir(class_path):
            if not npy_file.endswith('.npy'):
                continue
            data = np.load(os.path.join(class_path, npy_file))
            X.append(data)
            y.append(class_name)
    return np.array(X), np.array(y)

X_train, y_train = load_split(output_split_folder, 'train', VALID_CLASSES)
X_val,   y_val   = load_split(output_split_folder, 'val',   VALID_CLASSES)
X_test,  y_test  = load_split(output_split_folder, 'test',  VALID_CLASSES)

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')

# ── 2. Encode Labels ──────────────────────────────────────────────────────────
le = LabelEncoder()
le.fit(y_train)

y_train_enc = to_categorical(le.transform(y_train))
y_val_enc   = to_categorical(le.transform(y_val))
y_test_enc  = to_categorical(le.transform(y_test))

num_classes = len(le.classes_)
print(f'Classes:       {num_classes}')
print(f'Random chance: {1/num_classes*100:.1f}%')

# ── 3. Build Model ────────────────────────────────────────────────────────────
model = Sequential([
    LSTM(32, input_shape=(TARGET_FRAMES, 126)),  # single LSTM, smaller
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.summary()

# ── 4. Compile ────────────────────────────────────────────────────────────────
model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── 5. Train ──────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train_enc,
    validation_data=(X_val, y_val_enc),
    epochs=200,
    batch_size=16,
    callbacks=[early_stop]
)

# ── 6. Evaluate ───────────────────────────────────────────────────────────────
loss, accuracy = model.evaluate(X_test, y_test_enc)
print(f'\nTest Accuracy: {accuracy * 100:.2f}%')
print(f'Random Chance: {1/num_classes*100:.1f}%')

# ── 7. Per Class Report ───────────────────────────────────────────────────────
y_pred         = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test_enc, axis=1)

print('\n=== Per Class Report ===')
print(classification_report(
    y_test_classes,
    y_pred_classes,
    target_names=le.classes_
))

X_train: (477, 10, 126)
X_val:   (35, 10, 126)
X_test:  (34, 10, 126)
Classes:       30
Random chance: 3.3%


C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_27 (LSTM)                       │ (None, 32)                  │          20,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_27 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_27 (Dense)                     │ (None, 30)                  │             990 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 21,342 (83.37 KB)

 Trainable params: 21,342 (83.37 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.0189 - loss: 3.4119 - val_accuracy: 0.0286 - val_loss: 3.3627
Epoch 2/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0524 - loss: 3.3391 - val_accuracy: 0.0571 - val_loss: 3.3047
Epoch 3/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0587 - loss: 3.2934 - val_accuracy: 0.0857 - val_loss: 3.2474
Epoch 4/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0943 - loss: 3.2186 - val_accuracy: 0.0857 - val_loss: 3.1959
Epoch 5/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1111 - loss: 3.1594 - val_accuracy: 0.1143 - val_loss: 3.1618
Epoch 6/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1069 - loss: 3.1056 - val_accuracy: 0.1143 - val_loss: 3.1036
Epoch 7/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1069 - loss: 3.0335 - val_accuracy: 0.0857 - val_loss: 3.0471
Epoch 8/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1551 - loss: 2.9981 - val_accuracy: 0.

C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
C:\Users\justi\MachineLearning\Project\asl_env\lib\

In [29]:

# ── Random Seed ───────────────────────────────────────────────────────────────
D = 80
random.seed(D)
np.random.seed(D)
tf.random.set_seed(D)

# ── Config ────────────────────────────────────────────────────────────────────
output_split_folder = r'C:\Users\justi\MachineLearning\Project\dataset_split_mpLM_aug2'
TARGET_FRAMES = 10

VALID_CLASSES = [
    'computer', 'thin', 'trade', 'carrot', 'cool', 'who', 'accident',
    'basketball', 'before', 'bird', 'bowling', 'brother', 'change', 'cold',
    'corn', 'dark', 'family', 'hot', 'later', 'laugh', 'leave', 'new',
    'pizza', 'school', 'tall', 'thanksgiving', 'what', 'white', 'write', 'yes'
]

# ── 1. Load ALL Data (pool train/val/test together for CV) ────────────────────
def load_split(split_folder, split, valid_classes):
    X, y = [], []
    split_path = os.path.join(split_folder, split)
    for class_name in valid_classes:
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue
        for npy_file in os.listdir(class_path):
            if not npy_file.endswith('.npy'):
                continue
            data = np.load(os.path.join(class_path, npy_file))
            X.append(data)
            y.append(class_name)
    return np.array(X), np.array(y)

# Load all splits and combine
X_train, y_train = load_split(output_split_folder, 'train', VALID_CLASSES)
X_val,   y_val   = load_split(output_split_folder, 'val',   VALID_CLASSES)
X_test,  y_test  = load_split(output_split_folder, 'test',  VALID_CLASSES)

X_all = np.concatenate([X_train, X_val, X_test], axis=0)
y_all = np.concatenate([y_train, y_val, y_test], axis=0)

print(f'Total samples: {len(X_all)}')

# ── 2. Encode Labels ──────────────────────────────────────────────────────────
le = LabelEncoder()
y_all_enc = le.fit_transform(y_all)

num_classes = len(le.classes_)
print(f'Classes:       {num_classes}')
print(f'Random chance: {1/num_classes*100:.1f}%')

# ── 3. Cross Validation ───────────────────────────────────────────────────────
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=D)
fold_accuracies = []
all_y_true = []
all_y_pred = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(X_all, y_all_enc)):
    print(f'\n=== Fold {fold+1}/5 ===')

    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all_enc[train_idx], y_all_enc[test_idx]

    y_tr_cat = to_categorical(y_tr, num_classes=num_classes)
    y_te_cat = to_categorical(y_te, num_classes=num_classes)

    # ── Build Model ───────────────────────────────────────────────────────────
    model = Sequential([
        LSTM(32, input_shape=(TARGET_FRAMES, 126)),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )

    model.fit(
        X_tr, y_tr_cat,
        validation_split=0.2,
        epochs=200,
        batch_size=8,
        callbacks=[early_stop],
        verbose=0  # silent per epoch, just show fold results
    )

    _, acc = model.evaluate(X_te, y_te_cat, verbose=0)
    fold_accuracies.append(acc)

    # Collect predictions for final report
    y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
    all_y_true.extend(y_te)
    all_y_pred.extend(y_pred)

    print(f'  Fold {fold+1} accuracy: {acc*100:.2f}%')

# ── 4. Final Results ──────────────────────────────────────────────────────────
print(f'\n=== Cross Validation Results ===')
print(f'Per fold:      {[f"{a*100:.1f}%" for a in fold_accuracies]}')
print(f'Mean accuracy: {np.mean(fold_accuracies)*100:.2f}%')
print(f'Std:           {np.std(fold_accuracies)*100:.2f}%')
print(f'Random chance: {1/num_classes*100:.1f}%')

# ── 5. Per Class Report ───────────────────────────────────────────────────────
print('\n=== Per Class Report (aggregated across folds) ===')
print(classification_report(
    all_y_true,
    all_y_pred,
    target_names=le.classes_
))

Total samples: 546
Classes:       30
Random chance: 3.3%

=== Fold 1/5 ===


C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Fold 1 accuracy: 12.73%

=== Fold 2/5 ===
  Fold 2 accuracy: 3.67%

=== Fold 3/5 ===
  Fold 3 accuracy: 10.09%

=== Fold 4/5 ===
  Fold 4 accuracy: 11.01%

=== Fold 5/5 ===
  Fold 5 accuracy: 5.50%

=== Cross Validation Results ===
Per fold:      ['12.7%', '3.7%', '10.1%', '11.0%', '5.5%']
Mean accuracy: 8.60%
Std:           3.43%
Random chance: 3.3%

=== Per Class Report (aggregated across folds) ===
              precision    recall  f1-score   support

    accident       0.00      0.00      0.00        17
  basketball       0.00      0.00      0.00        17
      before       0.00      0.00      0.00        18
        bird       0.40      0.12      0.18        17
     bowling       0.00      0.00      0.00        17
     brother       0.11      0.17      0.13        18
      carrot       0.08      0.35      0.13        20
      change       0.00      0.00      0.00        18
        cold       0.11      0.24      0.15        17
    computer       0.08      0.08      0.08        2

C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [31]:
from collections import Counter
class_counts = Counter(y_train)
top_n = 10  # try 10, 15, 20
top_classes = [c for c, _ in class_counts.most_common(top_n)]

# 2. Retrain using only those classes
X_train_sub = X_train[np.isin(y_train, top_classes)]
y_train_sub = y_train[np.isin(y_train, top_classes)]

# Filter val and test to only top_classes
X_val_sub  = X_val[np.isin(y_val, top_classes)]
y_val_sub  = y_val[np.isin(y_val, top_classes)]
X_test_sub = X_test[np.isin(y_test, top_classes)]
y_test_sub = y_test[np.isin(y_test, top_classes)]

# Re-encode labels for only these classes
le_sub = LabelEncoder()
le_sub.fit(y_train_sub)

y_train_enc_sub = to_categorical(le_sub.transform(y_train_sub))
y_val_enc_sub   = to_categorical(le_sub.transform(y_val_sub))
y_test_enc_sub  = to_categorical(le_sub.transform(y_test_sub))

num_classes_sub = len(le_sub.classes_)
print(f'Classes: {num_classes_sub}')
print(f'Train: {X_train_sub.shape}')
print(f'Val:   {X_val_sub.shape}')
print(f'Test:  {X_test_sub.shape}')
print(f'Random chance: {1/num_classes_sub*100:.2f}%')

# Rebuild model for new number of classes
model_sub = Sequential([
        LSTM(32, input_shape=(TARGET_FRAMES, 126)),
        Dropout(0.3),
        Dense(num_classes_sub, activation='softmax')
    ])

model_sub.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model_sub.fit(
    X_train_sub, y_train_enc_sub,
    validation_data=(X_val_sub, y_val_enc_sub),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop]
)

loss, accuracy = model_sub.evaluate(X_test_sub, y_test_enc_sub)
print(f'\nTest Accuracy: {accuracy * 100:.2f}%')
print(f'Random chance: {1/num_classes_sub*100:.2f}%')

Classes: 10
Train: (177, 10, 126)
Val:   (15, 10, 126)
Test:  (10, 10, 126)
Random chance: 10.00%


C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 79ms/step - accuracy: 0.0904 - loss: 2.3467 - val_accuracy: 0.2000 - val_loss: 2.2430
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.1638 - loss: 2.2302 - val_accuracy: 0.2000 - val_loss: 2.1886
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2316 - loss: 2.1365 - val_accuracy: 0.2000 - val_loss: 2.1286
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2938 - loss: 2.0937 - val_accuracy: 0.1333 - val_loss: 2.0633
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2599 - loss: 2.0267 - val_accuracy: 0.2667 - val_loss: 1.9886
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2994 - loss: 1.9689 - val_accuracy: 0.2000 - val_loss: 1.9556
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3277 - loss: 1.9616 - val_accuracy: 0.3333 - val_loss: 1.9082
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3390 - loss: 1.8442 - val_accuracy: 0.

In [37]:

# ── Random Seed ───────────────────────────────────────────────────────────────
D = 80
random.seed(D)
np.random.seed(D)
tf.random.set_seed(D)

# ── Config ────────────────────────────────────────────────────────────────────
output_split_folder = r'C:\Users\justi\MachineLearning\Project\dataset_split_mpLM_aug2'
TARGET_FRAMES = 10
TOP_N         = 10

VALID_CLASSES = [
    'computer', 'thin', 'trade', 'carrot', 'cool', 'who', 'accident',
    'basketball', 'before', 'bird', 'bowling', 'brother', 'change', 'cold',
    'corn', 'dark', 'family', 'hot', 'later', 'laugh', 'leave', 'new',
    'pizza', 'school', 'tall', 'thanksgiving', 'what', 'white', 'write', 'yes'
]

# ── 1. Load Data ──────────────────────────────────────────────────────────────
def load_split(split_folder, split, valid_classes):
    X, y = [], []
    split_path = os.path.join(split_folder, split)
    for class_name in valid_classes:
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue
        for npy_file in os.listdir(class_path):
            if not npy_file.endswith('.npy'):
                continue
            data = np.load(os.path.join(class_path, npy_file))
            X.append(data)
            y.append(class_name)
    return np.array(X), np.array(y)

X_train, y_train = load_split(output_split_folder, 'train', VALID_CLASSES)
X_val,   y_val   = load_split(output_split_folder, 'val',   VALID_CLASSES)
X_test,  y_test  = load_split(output_split_folder, 'test',  VALID_CLASSES)

# ── 2. Get Top N Classes ──────────────────────────────────────────────────────
class_counts = Counter(y_train)
top_classes  = [c for c, _ in class_counts.most_common(TOP_N)]
print(f'Top {TOP_N} classes: {top_classes}')

# ── 3. Filter to Top N ────────────────────────────────────────────────────────
def filter_classes(X, y, classes):
    mask = np.isin(y, classes)
    return X[mask], y[mask]

X_train, y_train = filter_classes(X_train, y_train, top_classes)
X_val,   y_val   = filter_classes(X_val,   y_val,   top_classes)
X_test,  y_test  = filter_classes(X_test,  y_test,  top_classes)

# ── 4. Merge Val into Train ───────────────────────────────────────────────────
X_train = np.concatenate([X_train, X_val], axis=0)
y_train = np.concatenate([y_train, y_val], axis=0)

print(f'\nAfter merging val into train:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test:  {X_test.shape}')

# ── 5. Encode Labels ──────────────────────────────────────────────────────────
le = LabelEncoder()
le.fit(y_train)

y_train_enc = to_categorical(le.transform(y_train))
y_test_enc  = to_categorical(le.transform(y_test))

num_classes = len(le.classes_)
print(f'\nClasses:       {num_classes}')
print(f'Random chance: {1/num_classes*100:.1f}%')

# ── 6. Build Model ────────────────────────────────────────────────────────────
model = Sequential([
    LSTM(32, input_shape=(TARGET_FRAMES, 126)),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.summary()

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── 7. Train ──────────────────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train_enc,
    epochs=150,
    batch_size=8,
    verbose=1
)

# ── 8. Evaluate ───────────────────────────────────────────────────────────────
loss, accuracy = model.evaluate(X_test, y_test_enc, verbose=0)
print(f'\nTest Accuracy: {accuracy * 100:.2f}%')
print(f'Random Chance: {1/num_classes*100:.1f}%')

# ── 9. Per Class Report ───────────────────────────────────────────────────────
y_pred         = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test_enc, axis=1)

print('\n=== Per Class Report ===')
print(classification_report(
    y_test_classes,
    y_pred_classes,
    target_names=le.classes_
))

# ── 10. Per Class Sample Count ────────────────────────────────────────────────
print('=== Sample counts ===')
for cls in sorted(top_classes):
    train_n = np.sum(y_train == cls)
    test_n  = np.sum(y_test  == cls)
    print(f'  {cls}: train={train_n} test={test_n}')

Top 10 classes: [np.str_('computer'), np.str_('thin'), np.str_('trade'), np.str_('carrot'), np.str_('cool'), np.str_('who'), np.str_('accident'), np.str_('basketball'), np.str_('before'), np.str_('bird')]

After merging val into train:
  X_train: (192, 10, 126)
  X_test:  (10, 10, 126)

Classes:       10
Random chance: 10.0%


C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_28"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_40 (LSTM)                       │ (None, 32)                  │          20,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_40 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_40 (Dense)                     │ (None, 10)                  │             330 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 20,682 (80.79 KB)

 Trainable params: 20,682 (80.79 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1562 - loss: 2.2889
Epoch 2/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2240 - loss: 2.1135
Epoch 3/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2500 - loss: 1.9745
Epoch 4/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2552 - loss: 1.9382
Epoch 5/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3229 - loss: 1.8519
Epoch 6/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3021 - loss: 1.8415
Epoch 7/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3177 - loss: 1.7825
Epoch 8/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3490 - loss: 1.7135
Epoch 9/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3594 - loss: 1.6427
Epoch 10/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4323 - loss: 1.5876
Epoch 11/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4010 - loss: 1.5628
Epoch 12/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy

C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\justi\MachineLearning\Project\asl_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri